# 08. Phase Segmentation Test

Pipeline step ⑥. Verifies `segment_phases()` on the synthetic squat baseline.

Phase segmentation detects the intra-rep kinematic turn-around point from the
smoothed reference-landmark trajectory and writes kinematic phase labels
(`Descent` / `Ascent` / `Bottom_Hold`) to the `phase` column.

Pipeline position: Normalization → **Phase Segmentation** → Motion Attribution

This notebook assumes that the following notebooks are already passing:
- 00_environment_check through 07_preprocessing_test

Reference: `docs/code_revision_plan.md` §1, `docs/terminology.md` §5 (Kinematic phase)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig,
    ExerciseDefinitionConfig,
    NormalizationConfig,
    PhaseSegmentationConfig,
    PipelineConfig,
    ValidationConfig,
    run_pipeline,
)
from movement.segmentation import PhaseSegmentationReport, segment_phases
from movement.validation import run_basic_validation
from movement.config import make_coordinate_columns, make_required_columns, make_visibility_columns

print('imports OK')

## Data Setup

Mirrors pipeline order ①–⑤ to produce a normalized dataframe with annotation and exercise definition.

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)

run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)

ann_df = load_annotation_csv(ann_path)
df_ann, _ = apply_annotation(df_raw, ann_df)

exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
assert exercise_def.phase_segmentation is not None, 'squat.yaml must have phase_segmentation block'

df_norm, _ = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)

print(f'normalized: {df_norm.shape[0]} frames')
print(f'rep frames: {(df_norm["segment_type"] == "rep").sum()}')
print(f'phase_segmentation spec: {exercise_def.phase_segmentation}')

## Direct segment_phases() Test

In [ ]:
df_seg, reports = segment_phases(df_norm, exercise_def, fps_default=30.0)

print(f'output shape: {df_seg.shape}')
print(f'PhaseSegmentationReport count: {len(reports)}')
for r in reports:
    print(f'  rep {r.rep_id}: inflection_frames={r.inflection_frames}  phase_assignments={r.phase_assignments}')

## Check 1: `phase` column populated for rep frames

In [ ]:
assert 'phase' in df_seg.columns, 'phase column missing'

rep_mask  = df_seg['segment_type'] == 'rep'
nrep      = rep_mask.sum()
n_labeled = df_seg.loc[rep_mask, 'phase'].notna().sum()
print(f'rep frames: {nrep}  labeled: {n_labeled}')
assert n_labeled == nrep, f'Expected all {nrep} rep frames labeled, got {n_labeled}'
print('PASS: all rep frames have a phase label')

## Check 2: Kinematic labels are Descent / Ascent / Bottom_Hold

In [ ]:
valid_labels = {'Descent', 'Ascent', 'Bottom_Hold'}
actual = set(df_seg.loc[rep_mask, 'phase'].dropna().unique())
print(f'labels found: {actual}')
assert actual.issubset(valid_labels), f'unexpected labels: {actual - valid_labels}'
assert 'Descent' in actual, 'Descent label missing'
assert 'Ascent'  in actual, 'Ascent label missing'
print('PASS: all phase labels are valid kinematic labels')

counts = df_seg.loc[rep_mask, 'phase'].value_counts()
for label, cnt in counts.items():
    print(f'  {label}: {cnt} frames')

## Check 3: Non-rep frames remain NA

In [ ]:
non_rep_mask  = df_seg['segment_type'] != 'rep'
n_non_rep     = non_rep_mask.sum()
n_non_rep_na  = df_seg.loc[non_rep_mask, 'phase'].isna().sum()
print(f'non-rep frames: {n_non_rep}  still NA: {n_non_rep_na}')
assert n_non_rep_na == n_non_rep, 'non-rep frames must have NA phase'
print('PASS: non-rep frames remain NA')

## Check 4: PhaseSegmentationReport structure

In [ ]:
for r in reports:
    assert isinstance(r, PhaseSegmentationReport)
    assert isinstance(r.rep_id,                int)
    assert isinstance(r.inflection_frames,      list) and len(r.inflection_frames) >= 1
    assert isinstance(r.phase_assignments,      dict) and len(r.phase_assignments) >= 2
    assert r.rejected_reason is None,                   f'rep {r.rep_id} rejected: {r.rejected_reason}'
    d = r.as_dict()
    assert 'rep_id'             in d
    assert 'inflection_frames'  in d
    assert 'phase_assignments'  in d
    assert 'smoothing_method'   in d
print(f'PASS: PhaseSegmentationReport structure valid for all {len(reports)} reps')
print(json.dumps(reports[0].as_dict(), indent=2))

## Check 5: Inflection count matches multi_inflection_policy=global_extremum

In [ ]:
# global_extremum policy → exactly 1 inflection per rep (the global minimum of norm_z)
for r in reports:
    n_inf = len(r.inflection_frames)
    assert n_inf == 1, f'rep {r.rep_id}: expected 1 inflection, got {n_inf}'
print(f'PASS: each rep has exactly 1 inflection (global_extremum policy)')

## Check 6: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation         = ValidationConfig(enabled=True)
cfg.annotation         = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization      = NormalizationConfig(enabled=True)
cfg.phase_segmentation = PhaseSegmentationConfig(enabled=True, fps_default=30.0)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS)

assert 'phase_segmentation' in pipe_report, 'phase_segmentation key missing in report'
ps_report = pipe_report['phase_segmentation']
assert len(ps_report) >= 1, 'no reps processed'

rep_mask_pipe = pipe_df['segment_type'] == 'rep'
n_labeled_pipe = pipe_df.loc[rep_mask_pipe, 'phase'].notna().sum()
assert n_labeled_pipe == rep_mask_pipe.sum()
print(f'PASS: pipeline ⑥ phase_segmentation report: {len(ps_report)} reps')
print(f'      phase column labeled for {n_labeled_pipe} rep frames')
print(f'steps executed: {list(pipe_report.keys())}')

## Interpretation

Expected results for `mediapipe_squat_synthetic.csv` (clean baseline):

| Check | Expected |
|---|---|
| All rep frames labeled | `Descent` or `Ascent` or `Bottom_Hold` |
| Non-rep frames | NA |
| Inflections per rep | 1 (global minimum of hip-center norm_z) |
| `rejected_reason` | None (all reps successfully segmented) |
| `multi_inflection_collapsed` | False (clean data, single inflection) |

**Kinematic vs kinetic distinction**: `Descent` / `Ascent` / `Bottom_Hold` are
trajectory-direction labels, not muscle-action labels (eccentric / concentric).
They describe where the hip center moves, not the underlying muscle mechanics.